In [ ]:
# DATA CLEANING — ICMM Global Mining Dataset v1.5 (Latin America)
#
# Goal: take the ICMM Global Mining Dataset (July 2026) from our GitHub repo, keep the sites in Latin America and the Caribbean, clean them, and save two clean tables in the clean/ folder:
# - icmm_sites_latam.csv — one row per site (the main table);
# - icmm_site_commodities.csv — one row per site and mineral (a site that produces copper, gold and silver has 3 rows);
# plus icmm_mining_sites_clean.xlsx with both tables, a guide to every column, and the cleaning log.
#
# Project question: Do resource-rich countries in Latin America see more killings of environmental defenders?
#
# Role of this dataset: it shows where legal, large-scale mining is: mines, smelters, refineries and processing plants, for every mineral (critical minerals and gold) and every Latin American country.
#
# Sheet — What it contains
# About — licence, sources, and ICMM's definitions of every column, of the confidence levels and of the site types
# Global Mining Dataset v1.5 — one row per site (7,104 worldwide): name, coordinates, country, type of site, minerals, confidence. This is the sheet we clean.
# Supplement Sheet — summary counts by country and mineral (not used)
#
# Source: ICMM Global Mining Dataset, July 2026 (v1.5), global-mining-dataset-1-5.xlsx in github.com/len-chou/TeamTokio---Week-3-Project-, CC BY 4.0. Recommended citation: "ICMM Global Mining Dataset, July 2026 (V1.5) release."
# ICMM built it from: the WAPHA Global Metal Mines Database (Hudson-Edwards et al. 2023), Global Energy Monitor's coal and iron-ore trackers, an open database of coal and metal mines (Jasansky et al. 2024), the Skarn Associates database, and checks by companies (Vale, Lundin Mining…).
#
# Why cleaning is needed: some names have broken letters (an encoding error: "RÃƒÂ­o Turbio" instead of "Río Turbio"), one latitude is stored as text, some text has extra spaces, Mexico and Central America are put in "North America", the same type of site is written in several ways, mineral names have a typo and several names for the same mineral, and the list of other names of each site repeats the same name many times.
#
# Know before using it:
# - It is a snapshot of sites in 2026. There are no dates, so it cannot show mining growing over time.
# - ICMM counts as a "mine" a site where extraction "occurs or has occurred": some mines may be closed.
# - It gives no production or size: a giant copper mine and a small silver mine each count as one site.
# - About 7,100 sites that came only from S&P Global were not shared (licence), so the list is incomplete and may be uneven across countries.
# - It covers large-scale mining: small-scale and illegal mining is almost never in it.
# - Each site has a confidence level (explained in step 3F). A quarter of the Latin American sites are "Very Low".
#
# Units: one row = one site. Latitude and longitude in decimal degrees (WGS 84). Distances in kilometres.
#
# Every cleaning decision is recorded in log and saved at the end (sheet icmm_cleaning_log).

In [ ]:
# 1) Setup
#
# What:
# - import the libraries: pandas (tables), numpy (maths and yes/no choices), re (text patterns), Path (folders);
# - the link to the file in our GitHub repo, and the clean/ folder where results are saved;
# - note(): a small function that writes each cleaning decision into the list log (step, table, what was done, how many rows);
# - LATAM: the countries of the project with their ISO3 code and region; EXTRA: three places in the region that are not in the project list;
# - haversine_km(): the distance in km between two points on Earth (used in step 3F to find possible duplicate sites).
#
# Why: everything used in several steps is defined once, here.
#
# Output: nothing is printed.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)

# Read the file straight from our GitHub repo, so everyone in the team cleans exactly the same file
url = "https://raw.githubusercontent.com/len-chou/TeamTokio---Week-3-Project-/refs/heads/main/global-mining-dataset-1-5.xlsx"
SHEET = "Global Mining Dataset v1.5"
OUT_DIR = Path("clean")
OUT_DIR.mkdir(exist_ok=True)

log = []  # cleaning log: one row per decision
def note(step, table, action, n=None):
    log.append({"step": step, "table": table, "action": action, "rows_affected": n})

# Countries of the project: name, ISO3 code, region
LATAM = pd.DataFrame([
    ("Argentina", "ARG", "South America"), ("Bolivia", "BOL", "South America"),
    ("Brazil", "BRA", "South America"), ("Chile", "CHL", "South America"),
    ("Colombia", "COL", "South America"), ("Ecuador", "ECU", "South America"),
    ("Paraguay", "PRY", "South America"), ("Peru", "PER", "South America"),
    ("Uruguay", "URY", "South America"), ("Venezuela", "VEN", "South America"),
    ("Guyana", "GUY", "South America"), ("Suriname", "SUR", "South America"),
    ("Mexico", "MEX", "Mexico & Central America"), ("Guatemala", "GTM", "Mexico & Central America"),
    ("Honduras", "HND", "Mexico & Central America"), ("El Salvador", "SLV", "Mexico & Central America"),
    ("Nicaragua", "NIC", "Mexico & Central America"), ("Costa Rica", "CRI", "Mexico & Central America"),
    ("Panama", "PAN", "Mexico & Central America"), ("Belize", "BLZ", "Mexico & Central America"),
    ("Cuba", "CUB", "Caribbean"), ("Dominican Republic", "DOM", "Caribbean"), ("Haiti", "HTI", "Caribbean"),
], columns=["country", "iso3", "subregion"])
# In the region but not in the project list: kept, and marked in_latam_list = False
EXTRA = pd.DataFrame([("French Guiana", "GUF", "South America"), ("Jamaica", "JAM", "Caribbean"),
                      ("Trinidad and Tobago", "TTO", "Caribbean")], columns=["country", "iso3", "subregion"])
REGION_COUNTRIES = pd.concat([LATAM, EXTRA], ignore_index=True)

def haversine_km(lat1, lon1, lat2, lon2):
    """Distance in km between two points given in decimal degrees (works on single numbers or whole columns)."""
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))     # 6,371 km = radius of the Earth

In [2]:
# 2) Load and profile
#
# What: read the site sheet from GitHub and describe every column.
#
# How: profile() builds a small table with, for each column: its type (dtype), how many values are empty (missing, missing_%), how many different values it has (n_unique), and one example.
#
# How to read it:
# - ICMMID has 7,104 different values for 7,104 rows: every site has its own ID.
# - Latitude is "object" (text) while Longitude is a number: one latitude is broken (step 3A).
# - Group Names, Secondary Commodity and Other Commodities are often empty: normal, not every site has other names or more than one mineral.
# - The printed column name 'Secondary Commodity ' ends with a hidden space.
#
# Output: the sheet names, the size of the world table, the column name with a hidden space, and the profile.

In [3]:
xl = pd.ExcelFile(url)
print(xl.sheet_names)
raw = pd.read_excel(xl, SHEET)
print("world:", raw.shape)
print("column names with a hidden space:", [repr(c) for c in raw.columns if c != c.strip()])

def profile(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_%": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
        "example": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
    })
profile(raw)

['About', 'Global Mining Dataset v1.5', 'Supplement Sheet']
world: (7104, 13)
column names with a hidden space: ["'Secondary Commodity '"]


,dtype,missing,missing_%,n_unique,example
ICMMID,str,0,0.0,7104,ICMM00026
Confidence Factor,str,0,0.0,3,Very Low
Contains,str,0,0.0,7104,WA27; WA29
Mine Name,str,0,0.0,7056,Pato
Group Names,str,3134,44.1,3970,Pato;Pato Consolidated Dredging Co.
Latitude,object,0,0.0,7086,7.60535
Longitude,float64,0,0.0,7087,-74.8011
Asset Type,str,0,0.0,29,Mine
Country,str,0,0.0,128,Colombia
Region,str,0,0.0,7,South America


In [4]:
# Problems found in the raw data (each one is fixed in the step shown)
#
# 1. A column name ends with a space ("Secondary Commodity "), and some values have extra spaces ("Australia ", "gold "). → 3A
# 2. Some text has broken letters, an encoding error: "RÃƒÂ­o Turbio" for "Río Turbio", "CÃ´te d'Ivoire" for "Côte d'Ivoire". Some names were broken twice. → 3A
# 3. Latitude is stored as text because one value has a broken minus sign ("âˆ’23.8901"). → 3A
# 4. The "Region" column puts Mexico, Central America and the Caribbean in "North America" with the United States and Canada. → 3B
# 5. The type of site is written in different ways: "Mine;Smelter" and "Mine; Smelter", "Smelter;Refinery" and "Refinery; Smelter". → 3C
# 6. Group Names (the other names of a site) repeats the same name many times ("Cerro Moro;Cerro Moro;Cerro Moro") and once uses "*****" as a separator. → 3D
# 7. Mineral names: one typo ("cadmiuum"), one editing note left in the data ("remove"), two products that are not minerals ("calcine", "sulfuric acid"), and several names for one mineral ("chromite" / "chromium" / "ferrochrome"). → 3E
# 8. A few sites with the same name, in the same country, very close to each other: maybe one site entered twice. → 3G

In [5]:
# 3) Clean the site table
#
# Each of the following steps (3A–3H) fixes one or more of the problems above.

In [6]:
# 3A. Column names, spaces, broken letters, latitude
#
# What:
# 1. Column names: remove the hidden spaces, write them in lower case with "_" instead of spaces ("Secondary Commodity " → secondary_commodity), and give four columns shorter names (icmmid → icmm_id, mine_name → site_name, latitude → lat, longitude → lon). ICMM's "region" becomes region_icmm, because we make our own region in 3B.
# 2. Spaces: remove spaces before/after every text value, and turn double spaces into single spaces.
# 3. Broken letters: the function fix_text finds each broken piece of text (like "Ã­") and re-reads it with the right encoding ("í"). Some names were broken twice, so it repeats until nothing changes.
# 4. Latitude: after step 3 the broken minus becomes a real minus sign "−"; we replace it with "-" and turn the column into numbers.
#
# Why: "gold " and "gold" would count as two minerals; broken names cannot be read or matched; a latitude stored as text cannot be put on a map.
#
# Checks inside the cell (assert = stop with an error if it is not true): every coordinate is a number, inside -90/90 and -180/180, and every icmm_id is unique.
#
# Output: country names with accents after the repair (should show "Côte d'Ivoire" correctly).

In [7]:
df = raw.copy()

# 1. column names
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
df = df.rename(columns={"icmmid": "icmm_id", "latitude": "lat", "longitude": "lon", "mine_name": "site_name",
                        "region": "region_icmm"})
note("3A", "icmm", "column names to lower case with _ (removed the hidden space in 'Secondary Commodity ')", len(df.columns))

# 3. broken letters: a broken character = one letter from Â to ô followed by 1-3 special characters
BROKEN = re.compile("[\u00c2-\u00f4][\u0080-\u00bf\u0152\u0153\u0160\u0161\u0178\u017d\u017e"
                    "\u0192\u02c6\u02dc\u2010-\u203a\u20ac\u2122]{1,3}")

def fix_piece(match):
    """Re-read one broken piece with the right encoding ('Ã­' -> 'í'); leave it as it is if that fails."""
    for enc in ("cp1252", "latin-1"):
        try:
            return match.group(0).encode(enc).decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            continue
    return match.group(0)

def fix_text(text):
    """Repair every broken piece of a text; repeat, because some names were broken twice."""
    if not isinstance(text, str):
        return text
    for _ in range(4):
        fixed = BROKEN.sub(fix_piece, text)
        if fixed == text:
            break
        text = fixed
    return text

# 2. + 3. spaces and broken letters, in every text column
text_cols = [c for c in df.columns if c not in ("lon",)]
n_changed = 0
for c in text_cols:
    before = df[c].copy()
    df[c] = df[c].map(fix_text).map(lambda v: re.sub(r"\s{2,}", " ", v.strip()) if isinstance(v, str) else v)
    df[c] = df[c].replace("", pd.NA)
    n_changed += int((before.fillna("").astype(str) != df[c].fillna("").astype(str)).sum())
note("3A", "icmm", "removed extra spaces and repaired broken letters in text columns", n_changed)

# 4. latitude to numbers
df["lat"] = pd.to_numeric(df["lat"].astype(str).str.replace("\u2212", "-", regex=False), errors="coerce")
assert df[["lat", "lon"]].notna().all().all(), "a coordinate could not be read"
assert df["lat"].between(-90, 90).all() and df["lon"].between(-180, 180).all()
assert df["icmm_id"].is_unique
note("3A", "icmm", "latitude to numbers (1 value had a broken minus sign)", 1)

print(sorted(c for c in df["country"].unique() if not c.isascii()))

["Côte d'Ivoire"]


In [8]:
# 3B. Keep Latin America and the Caribbean, with our own region
#
# What:
# 1. a safety check: every country that ICMM puts in "South America" must be in our lists (so no country is lost by a spelling difference);
# 2. keep only the sites whose country is in LATAM or EXTRA;
# 3. add iso3 and subregion from our list (instead of ICMM's region);
# 4. in_latam_list = True for the project's countries, False for French Guiana, Jamaica and Trinidad and Tobago.
#
# Why: the project is about Latin America; ICMM's region mixes Mexico and Central America with the USA and Canada.
#
# Output: the number of sites and countries kept, the project countries with no ICMM site, and the number of sites per country.

In [9]:
unknown = sorted(set(df.loc[df["region_icmm"].eq("South America"), "country"]) - set(REGION_COUNTRIES["country"]))
assert not unknown, f"South American country missing from our list: {unknown}"

sites = df[df["country"].isin(REGION_COUNTRIES["country"])].copy()
sites = sites.merge(REGION_COUNTRIES, on="country", how="left")
sites["in_latam_list"] = sites["country"].isin(LATAM["country"])
note("3B", "icmm", "kept Latin American and Caribbean countries; added iso3 and our own subregion", len(sites))

print(f"{len(sites)} sites in {sites['country'].nunique()} countries "
      f"({(~sites['in_latam_list']).sum()} of them in French Guiana, Jamaica, Trinidad and Tobago)")
print("Project countries with no ICMM site:", sorted(set(LATAM["country"]) - set(sites["country"])))
sites["country"].value_counts()

709 sites in 20 countries (7 of them in French Guiana, Jamaica, Trinidad and Tobago)
Project countries with no ICMM site: ['Belize', 'Costa Rica', 'El Salvador', 'Haiti', 'Paraguay', 'Uruguay']


country
Brazil                 196
Peru                   129
Mexico                 109
Chile                  102
Bolivia                 54
Argentina               29
Colombia                24
Venezuela               19
Cuba                     7
Suriname                 6
Ecuador                  6
Guyana                   4
Honduras                 4
Dominican Republic       4
Guatemala                4
French Guiana            3
Nicaragua                3
Jamaica                  3
Panama                   2
Trinidad and Tobago      1
Name: count, dtype: int64

In [10]:
# 3C. Type of site
#
# ICMM's site types (from the About sheet):
# - Mine: a site where minerals are, or were, extracted, with a visible mining footprint;
# - Smelter: extracts metals from ores or concentrates;
# - Refinery: purifies raw metals into high-purity products (99 % or more);
# - Plant: processes mined material directly (crushing, grinding, washing, leaching, flotation);
# - Steel Plant: makes steel.
# A site with several types (e.g. "Mine; Smelter") is one large operation with facilities within 2.5 km of each other.
#
# What:
# 1. asset_type: each combination is split into its parts, the parts are sorted alphabetically and joined with "; ", so each combination has one spelling ("Refinery; Smelter" and "Smelter;Refinery" become "Refinery; Smelter"). The original value is kept in asset_type_raw.
# 2. is_mine, is_smelter, is_refinery, is_plant, is_steel_plant: True / False for each type, so you can filter easily (e.g. all sites that include a mine).
# 3. asset_main: one simple label per site: "mine" if the site includes a mine, otherwise "smelter / refinery", "steel plant" or "processing plant".
#
# Output: the number of sites per cleaned type (16 spellings become 11).

In [11]:
parts = sites["asset_type"].str.split(r"\s*;\s*").apply(lambda l: sorted({p.strip() for p in l if p.strip()}))
sites["asset_type_raw"] = sites["asset_type"]
sites["asset_type"] = parts.str.join("; ")

TYPES = ["Mine", "Smelter", "Refinery", "Plant", "Steel Plant"]
unknown = set(parts.explode()) - set(TYPES)
assert not unknown, f"unknown site type: {unknown}"
for t in TYPES:
    sites["is_" + t.lower().replace(" ", "_")] = parts.apply(lambda l: t in l)

sites["asset_main"] = np.select(
    [sites["is_mine"], sites["is_smelter"] | sites["is_refinery"], sites["is_steel_plant"]],
    ["mine", "smelter / refinery", "steel plant"], default="processing plant")
note("3C", "icmm", "one spelling per site-type combination; added is_mine/is_smelter/... and asset_main",
     int((sites["asset_type"] != sites["asset_type_raw"]).sum()))

print("Spellings before:", sites["asset_type_raw"].nunique(), "| after:", sites["asset_type"].nunique())
sites["asset_type"].value_counts()

Spellings before: 16 | after: 11


asset_type
Mine                        609
Smelter                      30
Mine; Plant                  21
Refinery; Smelter            12
Mine; Smelter                 9
Plant                         8
Mine; Refinery; Smelter       7
Mine; Refinery                6
Steel Plant                   4
Refinery                      2
Plant; Refinery; Smelter      1
Name: count, dtype: int64

In [12]:
# 3D. Other names of each site (group_names)
#
# What ICMM says: sites within 2.5 km of each other, or within 15 km with very similar names, were merged into one site. group_names lists the names of all the merged records, so the same name often appears several times.
#
# What: split the list on ";" (and on "*****", used once), remove spaces, keep each name once in its original order, and join them again with "; ".
#
# Why: "Cerro Moro;Cerro Moro;Cerro Moro" says the same as "Cerro Moro"; the repeats make the column hard to read and to search.
#
# Output: how many sites had repeated names, how many lists changed in total (the others only got the same separator "; "), and a few examples before and after.

In [13]:
def unique_names(text):
    if not isinstance(text, str):
        return text
    names = [n.strip() for n in re.split(r";|\*{5}", text) if n.strip()]
    return "; ".join(dict.fromkeys(names))          # dict.fromkeys keeps the first copy of each name, in order

before = sites["group_names"].copy()
had_repeats = before.dropna().map(lambda t: len([n.strip() for n in re.split(r";|\*{5}", t) if n.strip()])
                                  != len(set(n.strip() for n in re.split(r";|\*{5}", t) if n.strip())))
sites["group_names"] = sites["group_names"].map(unique_names)
changed = before.fillna("") != sites["group_names"].fillna("")
note("3D", "icmm", "group_names: removed repeated names and unified the separator", int(changed.sum()))

print("Sites with repeated names:", int(had_repeats.sum()))
print("Sites whose group_names changed (repeats removed and/or separator unified):", int(changed.sum()))
examples = had_repeats[had_repeats].index
pd.DataFrame({"before": before[examples], "after": sites.loc[examples, "group_names"]}).head(8)

Sites with repeated names: 229
Sites whose group_names changed (repeats removed and/or separator unified): 401


,before,after
9,La Camorra;La Camorra,La Camorra
14,Marmato Mines;Cerro Del Burro;Marmato Echandia Mines;Mar...,Marmato Mines; Cerro Del Burro; Marmato Echandia Mines; ...
26,Campanilla;Nambija District;Campanilla,Campanilla; Nambija District
41,Vale Base Metals; Sossego;Sossego;Sossego;Sossego,Vale Base Metals; Sossego
48,Alfa Y Arpon;Los Negros Mine;Hualgayoc;Cerro Corona;Hual...,Alfa Y Arpon; Los Negros Mine; Hualgayoc; Cerro Corona; ...
49,Creporizinho;Creporizinho,Creporizinho
55,Sayapullo;Sayapullo;Sayaatoc,Sayapullo; Sayaatoc
59,Luz Angelica Coco Mine;Quiruvilca;Quiruvilca;Quiruvilca,Luz Angelica Coco Mine; Quiruvilca


In [14]:
# 3E. Minerals: one vocabulary, gold and critical-mineral flags
#
# ICMM gives up to three mineral columns: primary_commodity (the main mineral), secondary_commodity (the second one) and other_commodities (any others, separated by ";"). They are kept as they are, and we add clean columns next to them.
#
# What:
# 1. Long table (icmm_site_commodities): one row per site and mineral, with its role (primary / secondary / other).
# 2. One name per mineral (mineral):
#    - typo: cadmiuum → cadmium; "iron" → iron ore;
#    - ferro-alloys → their base metal: ferronickel → nickel, ferromanganese / ferrosilicon manganese → manganese, ferroniobium → niobium, ferrochrome → chromium, ferrosilicon → silicon;
#    - same mineral, several names: chromite → chromium; alumina, aluminium, bauxite → bauxite / aluminium; potassium oxide, potassium sulfate → potash; platinum, palladium, rhodium, PGE / PGM → PGM (platinum-group metals); lanthanides → rare earths.
# 3. Dropped: "remove" (an editing note), "calcine" and "sulfuric acid" (products of processing, not minerals).
# 4. If a mineral appears twice for one site (e.g. "iron ore" primary and "iron" secondary), it is kept once, with its most important role.
# 5. is_critical: True if the mineral is one of the minerals in the IEA Critical Minerals Dataset 2026 that our team uses (the list is written in CRITICAL below). Gold is not a critical mineral. Tin, antimony, iron ore, bauxite / aluminium and coal are not in the IEA list either, so they are not flagged as critical.
# 6. Back in the site table:
#    - mineral_primary: the clean name of the main mineral;
#    - minerals_all: every mineral of the site, clean names, separated by "; ";
#    - n_minerals: how many minerals;
#    - has_gold: True if gold is one of the site's minerals (main or not);
#    - has_critical: True if at least one of the site's minerals is critical;
#    - mineral_group: one label from the main mineral only: "gold", "critical mineral", "iron, steel & coal", or "other".
#
# Why: with several names for one mineral, counts per mineral would be wrong; the flags make it easy to select gold or critical-mineral sites.
#
# Output: every name that was changed, then the number of sites per mineral group and region.

In [15]:
MINERAL_KEY = {
    "cadmiuum": "cadmium", "iron": "iron ore",
    "ferronickel": "nickel", "ferromanganese": "manganese", "ferrosilicon manganese": "manganese",
    "ferroniobium": "niobium", "ferrochrome": "chromium", "chromite": "chromium", "ferrosilicon": "silicon",
    "alumina": "bauxite / aluminium", "aluminium": "bauxite / aluminium", "bauxite": "bauxite / aluminium",
    "potassium oxide": "potash", "potassium sulfate": "potash",
    "pge / pgm": "PGM", "platinum": "PGM", "palladium": "PGM", "rhodium": "PGM",
    "lanthanides": "rare earths",
}
DROP = {"remove", "calcine", "sulfuric acid"}
# Minerals of the IEA Critical Minerals Dataset 2026 (our IEA source), written with the names used here
CRITICAL = {"arsenic", "boron", "cadmium", "chromium", "cobalt", "copper", "gallium", "germanium", "graphite",
            "hafnium", "indium", "lead", "lithium", "magnesium", "manganese", "molybdenum", "nickel", "niobium",
            "PGM", "rare earths", "selenium", "silicon", "silver", "tantalum", "tellurium", "titanium",
            "tungsten", "vanadium", "zinc", "zirconium"}

# 1. long table: one row per site and mineral
long_rows = []
for role, col in [("primary", "primary_commodity"), ("secondary", "secondary_commodity"), ("other", "other_commodities")]:
    s = sites[["icmm_id", col]].dropna().copy()
    s["commodity_raw"] = s[col].str.split(r"\s*[;,]\s*")
    s = s.explode("commodity_raw")
    s["commodity_raw"] = s["commodity_raw"].str.strip()
    s = s[s["commodity_raw"].ne("")]
    s["role"] = role
    long_rows.append(s[["icmm_id", "role", "commodity_raw"]])
commod = pd.concat(long_rows, ignore_index=True)

# 3. drop the non-minerals
low = commod["commodity_raw"].str.lower()
n_drop = int(low.isin(DROP).sum())
commod = commod[~low.isin(DROP)].copy()

# 2. one name per mineral
low = commod["commodity_raw"].str.lower()
commod["mineral"] = low.map(MINERAL_KEY).fillna(low)

# 4. each mineral once per site, with its most important role
commod["role_rank"] = commod["role"].map({"primary": 0, "secondary": 1, "other": 2})
commod = commod.sort_values("role_rank").drop_duplicates(["icmm_id", "mineral"]).drop(columns="role_rank")

# 5. critical flag
commod["is_critical"] = commod["mineral"].isin(CRITICAL)
note("3E", "icmm_commodities", "long table; one name per mineral; dropped 'remove', 'calcine', 'sulfuric acid'", n_drop)

# 6. back to the site table
g = commod.groupby("icmm_id")
sites["mineral_primary"] = sites["icmm_id"].map(commod.query("role == 'primary'").set_index("icmm_id")["mineral"])
sites["minerals_all"] = sites["icmm_id"].map(g["mineral"].apply(lambda s: "; ".join(sorted(s))))
sites["n_minerals"] = sites["icmm_id"].map(g.size())
sites["has_gold"] = sites["icmm_id"].map(g["mineral"].apply(lambda s: "gold" in set(s))).fillna(False).astype(bool)
sites["has_critical"] = sites["icmm_id"].map(g["is_critical"].any()).fillna(False).astype(bool)
sites["mineral_group"] = np.select(
    [sites["mineral_primary"].eq("gold"), sites["mineral_primary"].isin(CRITICAL),
     sites["mineral_primary"].isin(["iron ore", "steel", "coal"])],
    ["gold", "critical mineral", "iron, steel & coal"], default="other")
assert sites["mineral_primary"].notna().all(), "a site has no main mineral after cleaning"
note("3E", "icmm", "added mineral_primary, minerals_all, n_minerals, has_gold, has_critical, mineral_group", len(sites))

print("Mineral names that were changed:")
display(commod.assign(low=commod["commodity_raw"].str.lower())
        .loc[lambda d: d["low"] != d["mineral"].str.lower(), ["commodity_raw", "mineral"]].value_counts().to_frame("rows"))
pd.crosstab(sites["mineral_group"], sites["subregion"], margins=True)

Mineral names that were changed:


,,rows
commodity_raw,mineral,
alumina,bauxite / aluminium,10
ferronickel,nickel,7
bauxite,bauxite / aluminium,5
chromite,chromium,5
aluminium,bauxite / aluminium,4
PGE / PGM,PGM,4
palladium,PGM,3
ferromanganese,manganese,2
iron,iron ore,2


subregion,Caribbean,Mexico & Central America,South America,All
mineral_group,,,,
critical mineral,8,61,259,328
gold,3,39,138,180
"iron, steel & coal",1,18,103,122
other,3,4,72,79
All,15,122,572,709


In [16]:
# 3F. Confidence
#
# ICMM's confidence levels (from the About sheet), based on where the site comes from:
# - High: the site is in several of ICMM's main sources, or comes from Skarn Associates or primary data;
# - Moderate: the site is in one reliable source only (Global Energy Monitor or the Jasansky et al. database);
# - Low: the site came only from S&P Global. These sites were all removed from the public file, so no Latin American site is "Low";
# - Very Low: the site is in the WAPHA database only (ICMM found more errors in it).
#
# What:
# 1. confidence_factor becomes an ordered category (Very Low < Low < Moderate < High), so it sorts in the right order;
# 2. confidence_score: the same as a number (Very Low = 0, Low = 1, Moderate = 2, High = 3);
# 3. reliable: True for High and Moderate sites. Use it to check that a result still holds without the Very Low sites.
#
# Output: the number of sites per country and confidence level, with the share of "Very Low" sites; then the countries where more than 40 % of sites are "Very Low".

In [17]:
CONF = ["Very Low", "Low", "Moderate", "High"]
assert set(sites["confidence_factor"]) <= set(CONF), set(sites["confidence_factor"]) - set(CONF)
sites["confidence_factor"] = pd.Categorical(sites["confidence_factor"], categories=CONF, ordered=True)
sites["confidence_score"] = sites["confidence_factor"].cat.codes
sites["reliable"] = sites["confidence_factor"] >= "Moderate"
note("3F", "icmm", "confidence_factor as ordered category; added confidence_score and reliable (High/Moderate)",
     int((~sites["reliable"]).sum()))

ct = pd.crosstab(sites["country"], sites["confidence_factor"], margins=True)
ct["share_very_low"] = (ct["Very Low"] / ct["All"]).round(2)
ct.sort_values("All", ascending=False)

confidence_factor,Very Low,Moderate,High,All,share_very_low
country,,,,,
All,182,174,353,709,0.26
Brazil,46,60,90,196,0.23
Peru,38,34,57,129,0.29
Mexico,9,20,80,109,0.08
Chile,24,25,53,102,0.24
Bolivia,37,9,8,54,0.69
Argentina,5,4,20,29,0.17
Colombia,4,8,12,24,0.17
Venezuela,7,3,9,19,0.37


In [18]:
# Bolivia: 69 % of its sites are "Very Low" — counts for Bolivia should be checked with reliable sites only
print("Countries where more than 40% of sites are 'Very Low' confidence:")
ct.query("share_very_low > 0.4 and All >= 5")[["All", "Very Low", "share_very_low"]]

Countries where more than 40% of sites are 'Very Low' confidence:


confidence_factor,All,Very Low,share_very_low
country,,,
Bolivia,54,37,0.69


In [19]:
# 3G. Coordinates and possible duplicate sites
#
# What:
# 1. coords_ok: True if the site lies inside a rough rectangle around its country (BBOX, in degrees). It catches coordinates typed in the wrong country or with a wrong sign.
# 2. Possible duplicates: ICMM already merged sites within 2.5 km (or 15 km with very similar names). We look for what is left: two sites with the same name (after removing words like "Mine" or "Mina") in the same country. If they are less than 25 km apart, both get possible_duplicate = True and duplicate_of gives the icmm_id of the other one.
#    They are kept, not deleted: they can be two parts of one operation (for example a mine listed once for tin and once for silver) or one site entered twice. Same-name sites far apart are different places (e.g. Maricunga gold and Maricunga lithium, 92 km).
#
# Output: the number of sites outside their country (should be 0), then every same-name pair with its distance in km.

In [20]:
# rough rectangles: lat_min, lat_max, lon_min, lon_max
BBOX = {"Argentina": (-55.1, -21.7, -73.6, -53.6), "Bolivia": (-22.9, -9.6, -69.7, -57.4),
        "Brazil": (-33.8, 5.3, -74.0, -34.7), "Chile": (-56.0, -17.5, -75.7, -66.4),
        "Colombia": (-4.3, 13.4, -79.1, -66.8), "Ecuador": (-5.0, 1.5, -81.1, -75.2),
        "Peru": (-18.4, 0.0, -81.4, -68.6), "Venezuela": (0.6, 12.2, -73.4, -59.8),
        "Guyana": (1.1, 8.6, -61.4, -56.4), "Suriname": (1.8, 6.1, -58.1, -53.9),
        "French Guiana": (2.1, 5.8, -54.6, -51.6), "Mexico": (14.5, 32.8, -118.5, -86.7),
        "Guatemala": (13.7, 17.9, -92.3, -88.2), "Honduras": (12.9, 16.5, -89.4, -83.1),
        "Nicaragua": (10.7, 15.1, -87.7, -82.7), "Panama": (7.2, 9.7, -83.1, -77.1),
        "Cuba": (19.8, 23.3, -85.0, -74.1), "Dominican Republic": (17.5, 19.95, -72.0, -68.3),
        "Jamaica": (17.7, 18.6, -78.4, -76.2), "Trinidad and Tobago": (10.0, 11.4, -61.95, -60.5)}
assert set(sites["country"]) <= set(BBOX), set(sites["country"]) - set(BBOX)
sites["coords_ok"] = [BBOX[c][0] <= la <= BBOX[c][1] and BBOX[c][2] <= lo <= BBOX[c][3]
                      for c, la, lo in zip(sites["country"], sites["lat"], sites["lon"])]
print("Sites outside their country:", int((~sites["coords_ok"]).sum()))

# same-name pairs in the same country, and their distance
sites["name_key"] = (sites["site_name"].str.lower()
                     .str.replace(r"\b(mine|mina|mining|project)\b", "", regex=True).str.strip())
pairs = sites.merge(sites, on=["country", "name_key"], suffixes=("", "_b")).query("icmm_id < icmm_id_b")
pairs["dist_km"] = haversine_km(pairs["lat"], pairs["lon"], pairs["lat_b"], pairs["lon_b"]).round(1)
close = pairs[pairs["dist_km"] < 25]

sites["possible_duplicate"] = sites["icmm_id"].isin(set(close["icmm_id"]) | set(close["icmm_id_b"]))
other_id = pd.concat([close.set_index("icmm_id")["icmm_id_b"], close.set_index("icmm_id_b")["icmm_id"]])
sites["duplicate_of"] = sites["icmm_id"].map(other_id.groupby(level=0).first())
note("3G", "icmm", "coords_ok (inside the country); flagged same-name sites < 25 km apart as possible_duplicate",
     int(sites["possible_duplicate"].sum()))

print(f"Same-name pairs: {len(pairs)} | closer than 25 km (flagged): {len(close)}")
pairs[["country", "site_name", "icmm_id", "icmm_id_b", "primary_commodity", "primary_commodity_b",
       "confidence_factor", "confidence_factor_b", "dist_km"]].sort_values("dist_km")

Sites outside their country: 0
Same-name pairs: 13 | closer than 25 km (flagged): 6


,country,site_name,icmm_id,icmm_id_b,primary_commodity,primary_commodity_b,confidence_factor,confidence_factor_b,dist_km
161,Bolivia,Colquiri Mine,ICMM01091,ICMM01094,tin,silver,Moderate,Very Low,2.8
177,Bolivia,Morococala Mine,ICMM01157,ICMM01160,tin,tin,Very Low,High,4.5
120,Peru,Carahuacra Mine,ICMM00695,ICMM00698,zinc,copper,High,Moderate,5.6
181,Bolivia,Huanuni,ICMM01162,ICMM01166,tin,tin,Very Low,Moderate,5.9
109,Peru,Santander,ICMM00656,ICMM00662,zinc,zinc,Moderate,High,8.4
456,Mexico,La Caridad,ICMM08439,ICMM14273,copper,copper,High,High,20.6
284,Chile,Los Bronces,ICMM01705,ICMM08476,copper,copper,High,High,39.6
480,Brazil,Paraopeba,ICMM08534,ICMM08639,iron ore,iron ore,High,High,43.3
470,Chile,Maricunga,ICMM08497,ICMM13644,gold,lithium,High,High,92.1
515,Chile,Santo Domingo,ICMM08761,ICMM14402,copper,copper,High,Moderate,159.0


In [21]:
# 3H. Final tables and checks
#
# What:
# 1. icmm_sites: keep the useful columns in a clear order (who → where → what type → which minerals → how sure → original ICMM values), and sort by country and site name.
# 2. icmm_commodities: the long table from 3E, with the country, coordinates and reliable flag of each site added.
# 3. Final checks, one per line (all should be True):
#    - every icmm_id appears once;
#    - every site has coordinates, and they are inside its country;
#    - every site has a main mineral;
#    - every site has an ISO3 code and a region;
#    - no broken letters are left in any text column;
#    - no repeated names are left in group_names.
#
# Output: the table of checks.

In [22]:
SITE_COLS = ["icmm_id", "site_name", "group_names", "country", "iso3", "subregion", "in_latam_list", "lat", "lon",
             "coords_ok", "asset_type", "asset_main", "is_mine", "is_smelter", "is_refinery", "is_plant", "is_steel_plant",
             "mineral_primary", "minerals_all", "n_minerals", "mineral_group", "has_gold", "has_critical",
             "confidence_factor", "confidence_score", "reliable", "possible_duplicate", "duplicate_of",
             "primary_commodity", "secondary_commodity", "other_commodities", "asset_type_raw", "contains"]
icmm_sites = sites[SITE_COLS].sort_values(["country", "site_name"]).reset_index(drop=True)
icmm_commodities = (commod.merge(icmm_sites[["icmm_id", "country", "iso3", "lat", "lon", "reliable"]], on="icmm_id")
                    .sort_values(["country", "icmm_id", "role"]).reset_index(drop=True))

text = icmm_sites.select_dtypes(include=["object", "string"])
checks = {
    "icmm_id unique": icmm_sites["icmm_id"].is_unique,
    "every site has coordinates": bool(icmm_sites[["lat", "lon"]].notna().all().all()),
    "every site inside its country": bool(icmm_sites["coords_ok"].all()),
    "every site has a main mineral": bool(icmm_sites["mineral_primary"].notna().all()),
    "every site has iso3 and region": bool(icmm_sites[["iso3", "subregion"]].notna().all().all()),
    "no broken letters left": not text.apply(lambda s: s.astype(str).str.contains(BROKEN)).any().any(),
    "no repeated names in group_names": bool(icmm_sites["group_names"].dropna().str.split("; ")
                                             .apply(lambda l: len(l) == len(set(l))).all()),
}
pd.Series(checks, name="passed").to_frame()

,passed
icmm_id unique,True
every site has coordinates,True
every site inside its country,True
every site has a main mineral,True
every site has iso3 and region,True
no broken letters left,True
no repeated names in group_names,True


In [23]:
# 4) Guide to the columns
#
# What: a table that explains every column of the clean tables. It is shown here and saved as the sheet column_guide in the Excel file, so anyone opening the file knows what each column means.
#
# Output: the guide.

In [24]:
column_guide = pd.DataFrame([
    ("icmm_sites_latam", "icmm_id", "ICMM's unique ID of the site"),
    ("icmm_sites_latam", "site_name", "name of the site (broken letters repaired)"),
    ("icmm_sites_latam", "group_names", "other names of the same site (records ICMM merged into it), each name once, separated by '; '"),
    ("icmm_sites_latam", "country", "country of the site"),
    ("icmm_sites_latam", "iso3", "3-letter country code"),
    ("icmm_sites_latam", "subregion", "South America / Mexico & Central America / Caribbean (our region, not ICMM's)"),
    ("icmm_sites_latam", "in_latam_list", "True for the project's countries; False for French Guiana, Jamaica, Trinidad and Tobago"),
    ("icmm_sites_latam", "lat, lon", "latitude and longitude in decimal degrees (for merged sites: the average of the merged records)"),
    ("icmm_sites_latam", "coords_ok", "True if the coordinates are inside the country's rectangle (all are)"),
    ("icmm_sites_latam", "asset_type", "type of site, one spelling per combination (e.g. 'Mine; Smelter')"),
    ("icmm_sites_latam", "asset_main", "one label per site: mine / smelter / refinery / steel plant / processing plant"),
    ("icmm_sites_latam", "is_mine ... is_steel_plant", "True if the site includes that type of facility"),
    ("icmm_sites_latam", "mineral_primary", "main mineral, clean name"),
    ("icmm_sites_latam", "minerals_all", "all minerals of the site, clean names, separated by '; '"),
    ("icmm_sites_latam", "n_minerals", "number of minerals"),
    ("icmm_sites_latam", "mineral_group", "from the main mineral: gold / critical mineral / iron, steel & coal / other"),
    ("icmm_sites_latam", "has_gold", "True if gold is one of the site's minerals"),
    ("icmm_sites_latam", "has_critical", "True if at least one mineral is in the IEA Critical Minerals Dataset 2026"),
    ("icmm_sites_latam", "confidence_factor", "ICMM's confidence: High / Moderate / Very Low (Low is not in the public file)"),
    ("icmm_sites_latam", "confidence_score", "the same as a number: Very Low 0, Low 1, Moderate 2, High 3"),
    ("icmm_sites_latam", "reliable", "True for High and Moderate sites"),
    ("icmm_sites_latam", "possible_duplicate", "True if another site with the same name is less than 25 km away in the same country"),
    ("icmm_sites_latam", "duplicate_of", "icmm_id of that other site"),
    ("icmm_sites_latam", "primary_commodity, secondary_commodity, other_commodities", "ICMM's original mineral columns (spaces removed)"),
    ("icmm_sites_latam", "asset_type_raw", "ICMM's original type of site"),
    ("icmm_sites_latam", "contains", "codes of the source records behind the site: WA = WAPHA database, SK = Skarn Associates, "
                                     "JA = Jasansky et al. 2024, GMI / GMC = Global Energy Monitor iron-ore / coal trackers, "
                                     "SP = S&P Global (used by ICMM to check sites), VA = Vale, LUN = Lundin Mining, ICDA = chromium association"),
    ("icmm_site_commodities", "icmm_id", "ID of the site (links to icmm_sites_latam)"),
    ("icmm_site_commodities", "role", "primary / secondary / other: where the mineral was listed"),
    ("icmm_site_commodities", "commodity_raw", "the mineral name as written by ICMM"),
    ("icmm_site_commodities", "mineral", "clean mineral name"),
    ("icmm_site_commodities", "is_critical", "True if the mineral is in the IEA Critical Minerals Dataset 2026"),
    ("icmm_site_commodities", "country, iso3, lat, lon, reliable", "copied from the site table"),
], columns=["table", "column", "meaning"])
column_guide

,table,column,meaning
0,icmm_sites_latam,icmm_id,ICMM's unique ID of the site
1,icmm_sites_latam,site_name,name of the site (broken letters repaired)
2,icmm_sites_latam,group_names,other names of the same site (records ICMM merged into i...
3,icmm_sites_latam,country,country of the site
4,icmm_sites_latam,iso3,3-letter country code
5,icmm_sites_latam,subregion,South America / Mexico & Central America / Caribbean (ou...
6,icmm_sites_latam,in_latam_list,True for the project's countries; False for French Guian...
7,icmm_sites_latam,"lat, lon",latitude and longitude in decimal degrees (for merged si...
8,icmm_sites_latam,coords_ok,True if the coordinates are inside the country's rectang...
9,icmm_sites_latam,asset_type,"type of site, one spelling per combination (e.g. 'Mine; ..."


In [ ]:
# 5) Save the clean files
#
# What: save each table as a CSV in the clean/ folder, and all of them in icmm_mining_sites_clean.xlsx (one sheet per table): the two clean tables, the column guide, and the cleaning log.
#
# How to read the cleaning log: one row per decision, with the step of this notebook, the table it changed, what was done, and how many rows (or columns) it affected.
#
# Output: the folder where the files were saved, then the cleaning log.

In [25]:
log_df = pd.DataFrame(log)
tables = {
    "icmm_sites_latam": icmm_sites,
    "icmm_site_commodities": icmm_commodities,
    "column_guide": column_guide,
    "icmm_cleaning_log": log_df,
}
for name, t in tables.items():
    t.to_csv(OUT_DIR / f"{name}.csv", index=False)
with pd.ExcelWriter(OUT_DIR / "icmm_mining_sites_clean.xlsx") as xw:
    for name, t in tables.items():
        t.to_excel(xw, sheet_name=name[:31], index=False)

print("Saved to", OUT_DIR.resolve())
log_df

Saved to /Users/lena/Documents/IRONHACK DATA /Ironhack Data Analysis/WEEK 3/Project/dataset/clean


,step,table,action,rows_affected
0,3A,icmm,column names to lower case with _ (removed the hidden sp...,13
1,3A,icmm,removed extra spaces and repaired broken letters in text...,158
2,3A,icmm,latitude to numbers (1 value had a broken minus sign),1
3,3B,icmm,kept Latin American and Caribbean countries; added iso3 ...,709
4,3C,icmm,one spelling per site-type combination; added is_mine/is...,52
5,3D,icmm,group_names: removed repeated names and unified the sepa...,401
6,3E,icmm_commodities,"long table; one name per mineral; dropped 'remove', 'cal...",3
7,3E,icmm,"added mineral_primary, minerals_all, n_minerals, has_gol...",709
8,3F,icmm,confidence_factor as ordered category; added confidence_...,182
9,3G,icmm,coords_ok (inside the country); flagged same-name sites ...,12


In [ ]:
# 6) What this data can and cannot show
#
# Question — What the clean data gives — Limit
# Where is large-scale mining? — 709 sites with coordinates, country, type of site and minerals (icmm_sites_latam) — Large-scale only: small-scale and illegal mining is almost never in it. Some "mines" may be closed.
# Which minerals? — Main mineral, all minerals, gold and critical-mineral flags; one row per site and mineral in icmm_site_commodities — "Critical" follows the minerals of the IEA Critical Minerals Dataset 2026.
# How much mining? — Number of sites — Counts, not production: a giant mine and a small one each count as one site.
# Change over time — Nothing: one 2026 snapshot — No dates in the file.
# How reliable? — confidence_factor, reliable, possible_duplicate — Check results with reliable sites only (Bolivia: 69 % Very Low). About 7,100 S&P-only sites are missing worldwide, so countries with 0 or few sites may be gaps (Paraguay, Uruguay, Costa Rica, El Salvador, Belize, Haiti have none).
#
# Citation: ICMM Global Mining Dataset, July 2026 (V1.5) release. CC BY 4.0.